# Tableau enrichi — Application des modèles IA

**Projet** : Gallica Images — Classification gravures sur bois vs cuivre  
**Date**   : Avril 2026

Ce notebook charge le CSV brut produit par `../documentation_similarite_salomon/02_collecte_similarite.ipynb`,
applique les modèles ResNet50 (v1/v2/v3 par défaut) sur les résultats de similarité,
et génère le tableau HTML enrichi avec les colonnes technique IA.

**Entrée**  : `resultats/csv/salomon_segmente.csv`  
             `modeles/bois_cuivre/resnet50_v1|v2|v3.pth`  
**Sortie**  : `resultats/csv/salomon_enrichi_segmente.csv`  
             `resultats/similarite/Tableau_html/tableau_salomon_enrichi_segmente.html`

**Pour ajouter une nouvelle version** : ajouter son chemin dans `MODELES`
et relancer — les colonnes s'ajoutent automatiquement au tableau.


---

## 1. Configuration

**Seule cellule à modifier** pour ajouter ou retirer une version de modèle.

In [1]:
import sys
sys.path.insert(0, "..")
from gallica_utils import charger_resnet, appliquer_modele_df, generer_tableau_html, DEVICE

# ── Versions à appliquer — ajouter/enlever selon les besoins ─
MODELES = {
    "v1": "../../modeles/bois_cuivre/resnet50_v1.pth",
    "v2": "../../modeles/bois_cuivre/resnet50_v2.pth",
    "v3": "../../modeles/bois_cuivre/resnet50_bois_cuivre_v3.0.0.pth",
}

CHEMIN_CSV_BRUT   = "../../resultats/csv/salomon_segmente.csv"
CHEMIN_CSV_ENRICH = "../../resultats/csv/salomon_enrichi_segmente.csv"
CHEMIN_HTML       = "../../resultats/similarite/tableau_salomon_enrichi_segmente.html"
# ─────────────────────────────────────────────────────────────


import pandas as pd
import os

device = DEVICE
print(f"Device  : {device}")
print(f"Modèles : {list(MODELES.keys())}")

Device  : cuda
Modèles : ['v1', 'v2', 'v3']


## 2. Chargement du CSV brut

In [2]:
df = pd.read_csv(CHEMIN_CSV_BRUT)
print(f"✓ CSV chargé : {df.shape[0]} lignes")
df.head(3)

✓ CSV chargé : 1200 lignes


,salomon_page,salomon_ark,score,titre,auteur,date,corpus,technique,categorie,genre,palette,chromatic_mode,link,result_ark
0,32,bfkfk34fxfr,0.9537,[Illustrations de Les Métamorphoses] / [Non id...,Ovide (0043 av. J.-C.-0017). Auteur du texte,1540,NaN,estampe,Bande dessinée,Représentations humaines / Scènes,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk34fft6
1,32,bfkfk34fxfr,0.9441,[Illustrations de Roland furieux] / [Non ident...,"Arioste, L' (1474-1533). Auteur du texte",1544,NaN,estampe,Bande dessinée,Représentations végétales,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk8hh9pt
2,32,bfkfk34fxfr,0.9432,"Figure del Vecchio Testamento , con versi tosc...",NaN,1554,NaN,estampe,Imagerie religieuse,Représentations humaines / Scènes,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk5r7bnn


## 3. Application des modèles

Pour chaque version définie dans `MODELES`, on charge le modèle,
on prédit la technique de toutes les illustrations et on ajoute
deux colonnes `technique_ia_vN` et `technique_ia_conf_vN`.

In [3]:
for version, chemin_pth in MODELES.items():
    col_classe = f"technique_ia_{version}"
    col_conf   = f"technique_ia_conf_{version}"

    # Vérifier si la colonne existe déjà — ne pas recalculer
    if col_classe in df.columns:
        print(f"  {version} — déjà présent, ignoré")
        continue

    if not os.path.exists(chemin_pth):
        print(f"  {version} — modèle introuvable : {chemin_pth}")
        continue

    print(f"\n── Application modèle {version} ──")
    modele = charger_resnet(chemin_pth, device)
    df     = appliquer_modele_df(df, modele, col_classe, col_conf, device)

    # Libérer la mémoire GPU avant le prochain modèle
    import gc, torch
    del modele
    torch.cuda.empty_cache()
    gc.collect()

print("\n✓ Tous les modèles appliqués")


── Application modèle v1 ──


✓ ResNet50 chargé : ../../modeles/bois_cuivre/resnet50_v1.pth



✓ technique_ia_v1 — {'bois': 1043, 'cuivre': 156, 'inconnu': 1}

── Application modèle v2 ──


✓ ResNet50 chargé : ../../modeles/bois_cuivre/resnet50_v2.pth



✓ technique_ia_v2 — {'bois': 1041, 'cuivre': 154, 'inconnu': 5}

── Application modèle v3 ──


✓ ResNet50 chargé : ../../modeles/bois_cuivre/resnet50_bois_cuivre_v3.0.0.pth



✓ technique_ia_v3 — {'bois': 678, 'cuivre': 519, 'inconnu': 3}

✓ Tous les modèles appliqués


## 4. Comparaison des versions

In [4]:
print("Résumé des prédictions par version :\n")
for version in MODELES.keys():
    col = f"technique_ia_{version}"
    if col in df.columns:
        print(f"  {version} : {df[col].value_counts().to_dict()}")

# Accord entre toutes les versions pour cuivre
versions_disponibles = [v for v in MODELES.keys()
                        if f"technique_ia_{v}" in df.columns]

if len(versions_disponibles) >= 2:
    print(f"\nAccord toutes versions — cuivre certain :")
    masque = (df[f"technique_ia_{versions_disponibles[0]}"] == "cuivre")
    for v in versions_disponibles[1:]:
        masque = masque & (df[f"technique_ia_{v}"] == "cuivre")
    print(f"  {masque.sum()} illustrations classées cuivre par toutes les versions")

Résumé des prédictions par version :

  v1 : {'bois': 1043, 'cuivre': 156, 'inconnu': 1}
  v2 : {'bois': 1041, 'cuivre': 154, 'inconnu': 5}
  v3 : {'bois': 678, 'cuivre': 519, 'inconnu': 3}

Accord toutes versions — cuivre certain :
  110 illustrations classées cuivre par toutes les versions


## 5. Sauvegarde du CSV enrichi

In [5]:
df.to_csv(CHEMIN_CSV_ENRICH, index=False)
print(f"✓ CSV enrichi sauvegardé : {CHEMIN_CSV_ENRICH}")
print(f"  Colonnes : {df.columns.tolist()}")

✓ CSV enrichi sauvegardé : ../../resultats/csv/salomon_enrichi_segmente.csv
  Colonnes : ['salomon_page', 'salomon_ark', 'score', 'titre', 'auteur', 'date', 'corpus', 'technique', 'categorie', 'genre', 'palette', 'chromatic_mode', 'link', 'result_ark', 'technique_ia_v1', 'technique_ia_conf_v1', 'technique_ia_v2', 'technique_ia_conf_v2', 'technique_ia_v3', 'technique_ia_conf_v3']


## 6. Génération du tableau HTML enrichi

In [6]:
# Les colonnes IA disponibles sont déduites automatiquement
versions_html = [v for v in MODELES.keys()
                 if f"technique_ia_{v}" in df.columns]

generer_tableau_html(
    df,
    CHEMIN_HTML,
    colonnes_ia=versions_html,
    port=8085
)

✓ Tableau généré : ../../resultats/similarite/tableau_salomon_enrichi_segmente.html
✓ URL : http://localhost:8085/tableau_salomon_enrichi_segmente.html


gio: http://localhost:8085/tableau_salomon_enrichi_segmente.html: Operation not supported


### Test v1 et v2 sur les nouvelles éditions :

In [ ]:
import os

DOSSIER_SEG = os.path.join(os.path.abspath("../.."), "data", "editions_ovide", "segmentees")

print("Dossiers segmentés disponibles :\n")
print(f"  {'Dossier':55s} {'Nb':>5}")
print("  " + "─" * 63)

total = 0
for dossier in sorted(os.listdir(DOSSIER_SEG)):
    chemin = os.path.join(DOSSIER_SEG, dossier)
    if not os.path.isdir(chemin) or "_flip" in dossier or "_couleur" in dossier:
        continue
    n = len([f for f in os.listdir(chemin) if f.endswith(".jpg")])
    print(f"  {dossier:55s} {n:>5}")
    total += n

print("  " + "─" * 63)
print(f"  {'TOTAL':55s} {total:>5}")

In [ ]:
# Nouvelles éditions — jamais vues par v1/v2
# Exclus : salomon, solis, wickram, de_passe, clein/savery
NOUVELLES_EDITIONS = {
    # Bois
    "bois_eskrich_rouille_lyon1556"          : "bois",
    "bois_leroy_gueynard_lyon1510"           : "bois",
    # Taille douce
    "cuivre_baur_sn_augsbourg1709"           : "cuivre",
    "cuivre_baur_sn_vienne1639"              : "cuivre",
    "cuivre_blanchin_berthelin_rouen1651"    : "cuivre",
    "cuivre_borcht_plantin_anvers1591"       : "cuivre",
    "cuivre_bouche_blaeu_amsterdam1702"      : "cuivre",
    "cuivre_briot_drobet_lyon1628"           : "cuivre",
    "cuivre_franco_giunta_venise1584"        : "cuivre",
    "cuivre_gaultier_guillemot_paris1610"    : "cuivre",
    "cuivre_gaultier_sn_paris1616"           : "cuivre",
    "cuivre_gaultier_veuveguillemot_paris1614": "cuivre",
    "cuivre_goltzius_goltzius_haarlem1589"   : "cuivre",
    "cuivre_ht_molin_lyon1697"               : "cuivre",  # 3 tomes fusionnes (v4) — anciennement t4/t5/t6
    "cuivre_isaac_langelier_paris1617"       : "cuivre",
    "cuivre_mathieu_langelier_paris1619"     : "cuivre",
    "cuivre_monconet_sommaville_paris1660"   : "cuivre",
    "cuivre_philippe_hackiana_leyde1670"     : "cuivre",
    "cuivre_tempesta_dejode_anvers1606"      : "cuivre",
    "cuivre_tempesta_jansonius_amsterdam1610": "cuivre",
    "cuivre_weyen_barbin_paris1669"          : "cuivre",
}

print(f"Nouvelles éditions pour le test :")
print(f"  Bois        : {sum(1 for v in NOUVELLES_EDITIONS.values() if v == 'bois')}")
print(f"  Taille douce: {sum(1 for v in NOUVELLES_EDITIONS.values() if v == 'cuivre')}")
print(f"  Total       : {len(NOUVELLES_EDITIONS)} éditions")

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path

RACINE        = os.path.abspath("../..")
DOSSIER_SEG   = os.path.join(RACINE, "data", "editions_ovide", "segmentees")
DOSSIER_MOD   = os.path.join(RACINE, "modeles", "bois_cuivre")

# ── Charger les deux modèles ──────────────────────────────────
def charger_modele(chemin_pth, n_classes=2):
    modele = models.resnet50(weights=None)
    modele.fc = torch.nn.Linear(modele.fc.in_features, n_classes)
    modele.load_state_dict(torch.load(chemin_pth, map_location="cpu"))
    modele.eval()
    return modele

modele_v1 = charger_modele(os.path.join(DOSSIER_MOD, "resnet50_v1.pth"))
modele_v2 = charger_modele(os.path.join(DOSSIER_MOD, "resnet50_v2.pth"))
print("✓ v1 et v2 chargés")

# ── Transforms ───────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

CLASSES = ["bois", "cuivre"]

# ── Prédiction sur une image ──────────────────────────────────
def predire(modele, chemin_img):
    img    = Image.open(chemin_img).convert("RGB")
    tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        logits = modele(tensor)
        proba  = torch.softmax(logits, dim=1)[0]
        pred   = torch.argmax(proba).item()
    return CLASSES[pred], round(proba[pred].item(), 4)

# ── Boucle sur toutes les nouvelles éditions ─────────────────
resultats = []

for dossier, label_reel in NOUVELLES_EDITIONS.items():
    chemin_dossier = os.path.join(DOSSIER_SEG, dossier)
    if not os.path.exists(chemin_dossier):
        print(f"  ⚠️  {dossier} — introuvable")
        continue

    images = [f for f in os.listdir(chemin_dossier) if f.endswith(".jpg")]
    print(f"  {dossier} — {len(images)} images...", end="\r")

    for img_nom in images:
        chemin_img    = os.path.join(chemin_dossier, img_nom)
        pred_v1, conf_v1 = predire(modele_v1, chemin_img)
        pred_v2, conf_v2 = predire(modele_v2, chemin_img)
        resultats.append({
            "edition"    : dossier,
            "image"      : img_nom,
            "label_reel" : label_reel,
            "pred_v1"    : pred_v1,
            "conf_v1"    : conf_v1,
            "correct_v1" : pred_v1 == label_reel,
            "pred_v2"    : pred_v2,
            "conf_v2"    : conf_v2,
            "correct_v2" : pred_v2 == label_reel,
        })

df_test = pd.DataFrame(resultats)
print(f"\n✓ {len(df_test)} images testées")

# ── Statistiques globales ─────────────────────────────────────
print("\n" + "=" * 55)
print("STATISTIQUES GLOBALES")
print("=" * 55)
print(f"  Images testées   : {len(df_test)}")
print(f"  Accuracy v1      : {df_test['correct_v1'].mean():.4f}")
print(f"  Accuracy v2      : {df_test['correct_v2'].mean():.4f}")
print(f"  Confiance moy v1 : {df_test['conf_v1'].mean():.4f}")
print(f"  Confiance moy v2 : {df_test['conf_v2'].mean():.4f}")

# ── Statistiques par édition ──────────────────────────────────
print("\n" + "=" * 55)
print("ACCURACY PAR EDITION")
print("=" * 55)
print(f"  {'Edition':50s} {'Label':12s} {'v1':>6} {'v2':>6}")
print("  " + "─" * 78)

stats = (df_test.groupby(["edition", "label_reel"])
         .agg(acc_v1=("correct_v1", "mean"),
              acc_v2=("correct_v2", "mean"),
              nb    =("image",       "count"))
         .reset_index()
         .sort_values("label_reel"))

for _, row in stats.iterrows():
    v1 = f"{row['acc_v1']:.2f}"
    v2 = f"{row['acc_v2']:.2f}"
    print(f"  {row['edition']:50s} {row['label_reel']:12s} {v1:>6} {v2:>6}")

# ── Sauvegarde ────────────────────────────────────────────────
chemin_csv = os.path.join(RACINE, "resultats", "csv", "test_nouvelles_editions.csv")
df_test.to_csv(chemin_csv, index=False)
print(f"\n✓ Résultats sauvegardés → {chemin_csv}")

In [6]:
# Test sur Goltzius couleur
dossier_couleur = os.path.join(DOSSIER_SEG, "cuivre_goltzius_goltzius_haarlem1589_couleur")
images = [f for f in os.listdir(dossier_couleur) if f.endswith(".jpg")]

print(f"Goltzius couleur — {len(images)} images\n")

correct_v1 = correct_v2 = 0
for img_nom in images:
    chemin_img = os.path.join(dossier_couleur, img_nom)
    pred_v1, conf_v1 = predire(modele_v1, chemin_img)
    pred_v2, conf_v2 = predire(modele_v2, chemin_img)
    if pred_v1 == "cuivre": correct_v1 += 1
    if pred_v2 == "cuivre": correct_v2 += 1

print(f"  Accuracy v1 : {correct_v1/len(images):.2f} ({correct_v1}/{len(images)})")
print(f"  Accuracy v2 : {correct_v2/len(images):.2f} ({correct_v2}/{len(images)})")

Goltzius couleur — 38 images

  Accuracy v1 : 1.00 (38/38)
  Accuracy v2 : 1.00 (38/38)


In [21]:
print("Dossiers bois :\n")
for d in sorted(os.listdir(DOSSIER_SEG)):
    if d.startswith("bois") and "_flip" not in d:
        chemin = os.path.join(DOSSIER_SEG, d)
        n = len([f for f in os.listdir(chemin) if f.endswith(".jpg")])
        print(f"  {d:50s} : {n}")

print("\nDossiers cuivre :\n")
for d in sorted(os.listdir(DOSSIER_SEG)):
    if d.startswith("cuivre") and "_flip" not in d and "_couleur" not in d:
        chemin = os.path.join(DOSSIER_SEG, d)
        n = len([f for f in os.listdir(chemin) if f.endswith(".jpg")])
        print(f"  {d:50s} : {n}")

Dossiers bois :

  bois_eskrich_rouille_lyon1556                      : 42
  bois_leroy_gueynard_lyon1510                       : 19
  bois_salomon_rouille_lyon1557                      : 161
  bois_solis_feyerabend_francfort1581                : 184
  bois_wickram_behem_mayence1545                     : 50

Dossiers cuivre :

  cuivre_baur_sn_augsbourg1709                       : 161
  cuivre_baur_sn_vienne1639                          : 125
  cuivre_blanchin_berthelin_rouen1651                : 17
  cuivre_borcht_plantin_anvers1591                   : 182
  cuivre_bouche_blaeu_amsterdam1702                  : 126
  cuivre_briot_drobet_lyon1628                       : 28
  cuivre_depasse_depasse_koln1602                    : 134
  cuivre_depasse_jansonius_arnhem1607                : 136
  cuivre_franco_giunta_venise1584                    : 15
  cuivre_gaultier_guillemot_paris1610                : 16
  cuivre_gaultier_sn_paris1616                       : 14
  cuivre_gaultier_veuveguil